In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
def get_data():

  # Load the csv
  train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Training.csv")
  test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Testing.csv")

  # Extract features
  features_train = train_df[["Volume", "Doors"]].to_numpy()
  features_test = test_df[["Volume", "Doors"]].to_numpy()

  # Extract the class lables
  classes_train = train_df["Style"].to_list()
  classes_test = test_df["Style"].to_list()

  # Scale the features using the training data
  scaler = MinMaxScaler()
  features_train = scaler.fit_transform(features_train)
  features_test = scaler.transform(features_test)


  #Give meaningful names to features and classes
  feature_names = ["Volume", "Doors"]
  class_names = ["Sedan", "SUV", "Jeep", "Pickup", "Van"]

  return features_train,classes_train,features_test,classes_test,feature_names,class_names

In [13]:
features_train,classes_train,features_test,classes_test,feature_names,class_names = get_data()
# feature_names,class_names,features_train,classes_train,features_test,classes_test
print(classes_train[:3])


['Sedan', 'SUV', 'Sedan']


In [14]:
# Initialize list to store accuracy
accuracy_rows = []

best_k = None
best_acc = -1
best_predictions = None
best_confidence = None

max_k = min(15, len(features_train))

# Loop through K values to find the best
for k in range(1, max_k + 1):

    knn = KNeighborsClassifier(n_neighbors = k)
    knn.fit(features_train, classes_train)

    preds = knn.predict(features_test)
    acc = accuracy_score(classes_test, preds)

    probs = knn.predict_proba(features_test)
    conf = np.max(probs, axis=1)

    accuracy_rows.append([k, acc])

    # If this K is better accuracy than the previus K, store its value
    if acc > best_acc:
        best_acc = acc
        best_k = k
        best_predictions = preds
        best_confidence = conf

# Display results
print("Best K =", best_k)
print("Best Accuracy =", best_acc)

Best K = 7
Best Accuracy = 0.6451612903225806


In [15]:
accuracy_df = pd.DataFrame(accuracy_rows, columns=["K", "Accuracy"])
accuracy_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Accuracy.csv", index=False)
accuracy_df

,K,Accuracy
0,1,0.612903
1,2,0.548387
2,3,0.612903
3,4,0.580645
4,5,0.612903
5,6,0.580645
6,7,0.645161
7,8,0.612903
8,9,0.612903
9,10,0.612903


In [16]:
# Add columns to Testing.csv
test_path = "/content/drive/MyDrive/Colab Notebooks/Testing.csv"
test_df = pd.read_csv(test_path)

test_df["Prediction"] = best_predictions
test_df["Confidence"] = best_confidence

test_df.to_csv(test_path, index=False)
test_df

,Volume,Doors,Style,Prediction,Confidence
0,135,4,Pickup,SUV,0.571429
1,94,4,Sedan,Sedan,0.714286
2,94,4,SUV,Sedan,0.714286
3,134,4,SUV,Pickup,0.571429
4,122,4,SUV,Sedan,0.714286
5,85,2,Sedan,Sedan,0.571429
6,115,4,Sedan,Sedan,0.714286
7,102,4,Sedan,Sedan,0.714286
8,101,4,Sedan,Sedan,0.857143
9,121,4,Sedan,Sedan,0.714286


In [17]:
def get_predictions(K,features_train,classes_train,features_test):

#----Initialize the K-NN Classifier
    knn = KNeighborsClassifier(n_neighbors=K)
#----Train the model
    knn.fit(features_train, classes_train)
#----Predict for the test data
    predictions = knn.predict(features_test)

    return knn,predictions

In [18]:
knn,predictions = get_predictions(3,features_train,classes_train,features_test)
knn,predictions

(KNeighborsClassifier(n_neighbors=3),
 array(['SUV', 'Sedan', 'Sedan', 'SUV', 'Sedan', 'Sedan', 'SUV', 'Sedan',
        'Sedan', 'Sedan', 'Sedan', 'SUV', 'SUV', 'SUV', 'SUV', 'Sedan',
        'SUV', 'Sedan', 'SUV', 'Sedan', 'SUV', 'Sedan', 'SUV', 'SUV',
        'Sedan', 'SUV', 'Sedan', 'Sedan', 'SUV', 'Sedan', 'SUV'],
       dtype='<U6'))

In [19]:
def print_predictions(knn,predictions,features_test,classes_test):

#----Check the probability (How sure is the model?)
    probability = knn.predict_proba(features_test)
    confidences = np.max(probability, axis=1)
#----Output the result for each test data
    for index, value in enumerate(classes_test):
        print(f"The {classes_test[index]} is classified as: {predictions[index]}")
        print(f"Confidence is {confidences[index]}")

#----Compute the accuracy
    accuracy = accuracy_score(classes_test,predictions)
    print(f"The accuracy is {accuracy}")

In [20]:
print_predictions(knn,predictions,features_test,classes_test)

The Pickup is classified as: SUV
Confidence is 0.6666666666666666
The Sedan is classified as: Sedan
Confidence is 1.0
The SUV is classified as: Sedan
Confidence is 1.0
The SUV is classified as: SUV
Confidence is 0.6666666666666666
The SUV is classified as: Sedan
Confidence is 0.6666666666666666
The Sedan is classified as: Sedan
Confidence is 0.6666666666666666
The Sedan is classified as: SUV
Confidence is 0.6666666666666666
The Sedan is classified as: Sedan
Confidence is 0.6666666666666666
The Sedan is classified as: Sedan
Confidence is 1.0
The Sedan is classified as: Sedan
Confidence is 0.6666666666666666
The SUV is classified as: Sedan
Confidence is 1.0
The SUV is classified as: SUV
Confidence is 0.6666666666666666
The SUV is classified as: SUV
Confidence is 1.0
The SUV is classified as: SUV
Confidence is 0.6666666666666666
The SUV is classified as: SUV
Confidence is 0.6666666666666666
The Pickup is classified as: Sedan
Confidence is 0.6666666666666666
The SUV is classified as: SUV
C